In [ ]:
import pandas as pd
import platform
import glob
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [ ]:
# Check the operating system - file location is different if its Windows or OS
if platform.system() == 'Windows':
    path = "G:/My Drive/EarthEngineData"
else:
    # For Holden's Mac
    path = "/Users/holden/Personal Projects/flame-flame-fruit/FireData"

files = glob.glob(path + "/*.csv")

In [ ]:
# Throw all the files into one large pandas dataframe (this took me 22m to run btw)
df_list = []
for file in files:
    # Best practice is probably to put this all in a try catch but it worked for me for now...
    df = pd.read_csv(file)
    
    # Only add some of the days with no fire, since with too many it will skew predictions (since fire is rare)
    no_fires = df[df['T21_max'] == 0].sample(frac=0.1, random_state=42) #choosing 10% - we can change this
    # Every day with fire
    fires = df[df['T21_max'] > 0]
    
    both = pd.concat([fires, no_fires])
    df_list.append(both)

#Combine all the dataframes into one
final_df = pd.concat(df_list, ignore_index=True) #ignore_index to reset the row numbers on each list
print("final dataset size: ", final_df.shape)

In [ ]:
# Parse grid cell x,y indices out of system:index (format: YYYYMMDD_x,y)
# These represent which 4x4km grid cell in Colorado the row belongs to
final_df[['grid_x', 'grid_y']] = final_df['system:index'].str.split('_').str[1].str.split(',', expand=True).astype(int)

# Extracts the month number from the date (1-12) and adds it as a new column
final_df['month'] = pd.to_datetime(final_df['date']).dt.month

# Extracts year from the date and adds it as a new column
# Not necessary but could be useful to see if fire danger is increasing over time as a side project
final_df['year'] = pd.to_datetime(final_df['date']).dt.year

# Creates a 0/1 column in case there is a fire
final_df['fire'] = (final_df['T21_max'] > 0).astype(int)


In [ ]:
# This just helps visualize the data frame's structure
print(final_df.columns.tolist())
final_df.head(5)

In [ ]:
#split the data into training and testing sets
train_df = final_df[final_df['year'] < 2021] #train on data before 2021
test_df = final_df[final_df['year'] >= 2021] #test on data from 2021 and after

#seperate them into inputs and outputs
col_drop = ['fire','system:index','date', '.geo', 'T21_max', 'T21_mean', 'T21_stdDev'] #these are the columns we can get rid of
X_tain = train_df.drop(columns=col_drop)
x_test = test_df.drop(columns=col_drop)
y_train = train_df['fire']
y_test = test_df['fire']

In [ ]:
RandForest = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42) 
RandForest.fit(X_tain, y_train)

In [ ]:
prediction = RandForest.predict(x_test)
print(classification_report(y_test, prediction))